In [1]:
# get faq retriever
# create list queries
# get documents
# evaluation result (use precesion@k, recall@k, mrr, nDCG)

In [2]:
import sys
sys.path.append("../")

In [3]:
# get faq retriever
from pathlib import Path

from html import escape

import pandas as pd
from IPython.display import HTML, display

from evaluation.eval import evaluate_retrieval, load_tests, context_relevance
from evaluation.classes import ContextRelevanceWrapper
from evaluation.reporting import export_retrieval_report_to_markdown
from retrieval.faq_retriever import search_faq_hybrid

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def retrieving(test, top_k):
    context = search_faq_hybrid(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    return nodes

In [5]:
def _node_metadata(item):
    if hasattr(item, "node"):
        return getattr(item.node, "metadata", {}) or {}
    return getattr(item, "metadata", {}) or {}


def _node_score(item):
    score = getattr(item, "score", None)
    return round(score, 4) if isinstance(score, (int, float)) else None


def _node_text(item):
    node = getattr(item, "node", item)
    if hasattr(node, "get_content"):
        return node.get_content()
    return getattr(node, "text", "") or ""


def _preview(text, max_len=180):
    cleaned = " ".join(str(text).split())
    if len(cleaned) <= max_len:
        return cleaned
    return cleaned[: max_len - 3] + "..."


def _render_metrics(metrics):
    metrics_html = "".join(
        f"""
        <div style='padding:12px 14px;border:1px solid #d0d7de;border-radius:10px;background:#f6f8fa;'>
            <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.04em;'>{escape(name)}</div>
            <div style='font-size:24px;font-weight:700;color:#24292f;'>{value:.3f}</div>
        </div>
        """
        for name, value in metrics.items()
    )
    display(HTML(f"<div style='display:grid;grid-template-columns:repeat(4, minmax(0, 1fr));gap:12px;margin:12px 0 18px;'>{metrics_html}</div>"))


def _render_question_report(test, nodes, metrics, index, total):
    relevant_docs = set(test.relevant_docs)
    doc_rows = []

    for rank, item in enumerate(nodes, start=1):
        metadata = _node_metadata(item)
        section_id = metadata.get("section_id")
        doc_rows.append(
            {
                "rank": rank,
                "hit": "yes" if section_id in relevant_docs else "no",
                "section_id": section_id,
                "title": metadata.get("section_title", "-"),
                "score": _node_score(item),
                "preview": _preview(_node_text(item)),
            }
        )

    header_html = f"""
    <div style='margin:28px 0 10px;'>
        <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.08em;'>Question {index}/{total}</div>
        <div style='font-size:22px;font-weight:700;margin:6px 0 10px;color:#24292f;'>{escape(test.question)}</div>
        <div style='padding:12px 14px;background:#fff8c5;border:1px solid #e3d279;border-radius:10px;'>
            <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.04em;margin-bottom:6px;'>Relevant docs</div>
            <div style='font-size:16px;font-weight:600;color:#24292f;'>{escape(str(test.relevant_docs))}</div>
        </div>
    </div>
    """
    display(HTML(header_html))
    display(pd.DataFrame(doc_rows))
    filtered_metrics = {
            key: value
            for key, value in metrics.items()
            if key.startswith("precision") or key.startswith("recall") or key == "mrr" or key.startswith("ndcg") or key == "context_relevance"
    }
    _render_metrics(filtered_metrics)
    return {
        "question": test.question,
        "relevant_docs": list(test.relevant_docs),
        "metrics": filtered_metrics,
        "doc_rows": doc_rows,
    }


async def evaluate_all_retrieval(limit=None, top_k=5, markdown_path=None):
    """Render a readable retrieval report for the FAQ notebook."""

    tests = load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        nodes = retrieving(test, top_k)
        
        metrics = evaluate_retrieval(test, nodes, k=top_k)
        
        result = await context_relevance(ContextRelevanceWrapper(question=test.question, contexts=[_node_text(item) for item in nodes]))
        metrics["context_relevance"] = result
        rows.append(metrics)
        question_report = _render_question_report(test, nodes, metrics, index=index, total=len(selected_tests))
        question_report.update({"index": index, "total": len(selected_tests)})
        question_reports.append(question_report)

    
    report_df = pd.DataFrame(rows)
    metric_columns = [column for column in report_df.columns if column.startswith(("precision", "recall", "ndcg")) or column in {"mrr", "context_relevance"}]
    summary_df = report_df[metric_columns].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_retrieval_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/08_faq_retrieving.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df

In [6]:
await evaluate_all_retrieval(markdown_path="../reports/faq_retrieving_evaluation.md")

,rank,hit,section_id,title,score,preview
0,1,yes,1,How to register with an email,1.0000,How to register with an email Open the registr...
1,2,no,2,How to register with Google or Facebook account,0.7572,How to register with Google or Facebook accoun...
2,3,no,6,Can I have more than one HopShop account,0.6783,"Can I have more than one HopShop account Yes, ..."
3,4,no,4,How to register with your phone number,0.6718,How to register with your phone number Open th...
4,5,no,49,How to provide shipping details?,0.5984,How to provide shipping details? If you have a...


,rank,hit,section_id,title,score,preview
0,1,yes,2,How to register with Google or Facebook account,1.0000,How to register with Google or Facebook accoun...
1,2,no,6,Can I have more than one HopShop account,0.3925,"Can I have more than one HopShop account Yes, ..."
2,3,no,1,How to register with an email,0.3479,How to register with an email Open the registr...
3,4,no,3,Signing in with a password and email address,0.3165,Signing in with a password and email address I...
4,5,no,75,How to find a pick-up point or parcel locker c...,0.2981,How to find a pick-up point or parcel locker c...


,rank,hit,section_id,title,score,preview
0,1,yes,3,Signing in with a password and email address,1.0000,Signing in with a password and email address I...
1,2,no,1,How to register with an email,0.5002,How to register with an email Open the registr...
2,3,no,2,How to register with Google or Facebook account,0.4137,How to register with Google or Facebook accoun...
3,4,no,51,How can I track a parcel?,0.2763,How can I track a parcel? You can track your p...
4,5,no,49,How to provide shipping details?,0.2745,How to provide shipping details? If you have a...


,rank,hit,section_id,title,score,preview
0,1,yes,4,How to register with your phone number,0.9981,How to register with your phone number Open th...
1,2,yes,4,How to register with your phone number,0.7469,"your account within 3 business days, and retur..."
2,3,no,1,How to register with an email,0.7295,How to register with an email Open the registr...
3,4,no,2,How to register with Google or Facebook account,0.7202,How to register with Google or Facebook accoun...
4,5,no,6,Can I have more than one HopShop account,0.5973,"Can I have more than one HopShop account Yes, ..."


,rank,hit,section_id,title,score,preview
0,1,yes,5,GDPR — how you can download your personal data...,1.0000,GDPR — how you can download your personal data...
1,2,no,7,How to close your HopShop account,0.4022,How to close your HopShop account You can end ...
2,3,no,8,Withdrawing from the agreement with HopShop,0.3430,Withdrawing from the agreement with HopShop
3,4,no,6,Can I have more than one HopShop account,0.3419,"Can I have more than one HopShop account Yes, ..."
4,5,no,11,What happens next,0.3291,"What happens next Usually, we consider the app..."


...

,rank,hit,section_id,title,score,preview
0,1,yes,48,How much does delivery cost?,0.9975,How much does delivery cost? The cost of deliv...
1,2,no,35,Determine the delivery cost,0.8214,Determine the delivery cost If the seller has ...
2,3,no,50,I am buying many products — which delivery met...,0.7170,I am buying many products — which delivery met...
3,4,no,78,I want to change the delivery day. What should...,0.5480,I want to change the delivery day. What should...
4,5,no,51,How can I track a parcel?,0.5263,How can I track a parcel? You can track your p...


,rank,hit,section_id,title,score,preview
0,1,yes,49,How to provide shipping details?,1.0000,How to provide shipping details? If you have a...
1,2,no,61,How to order a parcel with HopShop Delivery,0.6695,How to order a parcel with HopShop Delivery Wh...
2,3,no,77,I want to change the pick-up point or parcel l...,0.6501,I want to change the pick-up point or parcel l...
3,4,no,90,I have been waiting a long time for my order d...,0.6346,I have been waiting a long time for my order d...
4,5,no,51,How can I track a parcel?,0.6228,How can I track a parcel? You can track your p...


,rank,hit,section_id,title,score,preview
0,1,yes,50,I am buying many products — which delivery met...,0.9540,I am buying many products — which delivery met...
1,2,no,32,How to pay for orders made from multiple sellers,0.9094,How to pay for orders made from multiple selle...
2,3,no,34,How to choose a delivery option,0.7691,How to choose a delivery option If you buy at ...
3,4,no,48,How much does delivery cost?,0.6097,How much does delivery cost? The cost of deliv...
4,5,no,25,How to buy again all products from a previous ...,0.5953,How to buy again all products from a previous ...


,precision@5,recall@5,mrr,ndcg@5,context_relevance
mean,0.198,0.98,0.857,0.889,0.96


,question,relevant_docs,retrieved_docs,precision@5,recall@5,mrr,ndcg@5,context_relevance
0,How do I register for a HopShop account using ...,[1],"[1, 2, 6, 4, 49]",0.20,1.0,1.000000,1.000000,1.0
1,How can I sign up using my Google or Facebook ...,[2],"[2, 6, 1, 3, 75]",0.20,1.0,1.000000,1.000000,1.0
2,How can I enable signing in with my email addr...,[3],"[3, 1, 2, 51, 49]",0.20,1.0,1.000000,1.000000,1.0
3,How do I register for a HopShop account using ...,[4],"[4, 1, 2, 6]",0.25,1.0,1.000000,1.000000,1.0
4,How can I download my personal data from my Ho...,[5],"[5, 7, 8, 6, 11]",0.20,1.0,1.000000,1.000000,1.0
5,Am I allowed to have multiple HopShop accounts...,[6],"[6, 50, 40, 46, 17]",0.20,1.0,1.000000,1.000000,1.0
6,How do I close my HopShop account?,[7],"[7, 16, 6, 14, 2]",0.20,1.0,1.000000,1.000000,1.0
7,How can I withdraw from my agreement with HopS...,[8],"[9, 8, 10, 7, 14]",0.20,1.0,0.500000,0.630930,1.0
8,Under what conditions can I withdraw from my a...,[9],"[9, 8, 10, 12, 13]",0.20,1.0,1.000000,1.000000,1.0
9,How can I withdraw from my agreement and what ...,[10],"[10, 9, 15, 11, 13]",0.20,1.0,1.000000,1.000000,1.0
